In [1]:
import torch
import sys
sys.path.insert(0, "/root/vk_work")
import torch.nn.functional as F
import torchattacks
from torch.utils.data import Subset, DataLoader
from torch import nn

from src.model import CustomResNet

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class Clf(nn.Module):
    def __init__(self, model, simkin=True):
        super().__init__()
        self.model, self.simkin = model, simkin
        self.register_buffer("mean", MEAN)
        self.register_buffer("std", STD)
    def forward(self, x):
        x = (x - self.mean) / self.std
        return self.model(x, mode='clas') if self.simkin else self.model(x)

In [2]:
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset
from src.custom_datasets import STL10RGBDataset

t = transforms.ToTensor()
test = STL10RGBDataset(load_dataset("jxie/stl10")["test"], transform=t)
test_loader = DataLoader(test, batch_size=128, shuffle=False, num_workers=8)

/root/vk_work/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
def load(ckpt):
    m = CustomResNet().to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    return Clf(m, simkin=True).to(device).eval()

def noise_load(ckpt):
    m = CustomResNet(noise_sigma=0.5).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    return Clf(m, simkin=True).to(device).eval()

best = load("../models/ResNetSIMv2-30_best.pth")
mid = load("../models/ResNetSIMv2-15_best.pth")
froz = load("../models/ResNetSIMv2_frozen_best.pth")
noisy = noise_load("../models/ResNetSIM_frozen_noisy_best.pth")
soft = noise_load("../models/ResNetSIM_soft_noisy_best.pth")
soft_v2 = noise_load("../models/ResNetSIM_soft_noisy_v2_best.pth")
soft_v3 = noise_load("../models/ResNetSIM_soft_noisy_v3_best.pth")


In [4]:
sub = Subset(test, range(1000))
sub_loader = DataLoader(sub, batch_size=128, shuffle=False, num_workers=8)

def adv_acc(clf, loader, atk=None):
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if atk is not None:
            x = atk(x, y)
        with torch.no_grad():
            correct += (clf(x).argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total

for name, clf in [("frozen", froz), ("frozen_noisy", noisy), ("soft_noisy", soft), ("pretrain15", mid), ("pretrain30", best)]:
    print(f"\n{name}: clean {adv_acc(clf, sub_loader):.3f}")
    for eps in [1/255, 2/255, 4/255, 8/255]:
        f = adv_acc(clf, sub_loader, torchattacks.FGSM(clf, eps=eps))
        p = adv_acc(clf, sub_loader, torchattacks.PGD(
            clf, eps=eps, alpha=eps/4, steps=10, random_start=True))
        print(f"  eps={eps*255:.0f}/255  FGSM {f:.3f}  PGD {p:.3f}")


frozen: clean 0.909
  eps=1/255  FGSM 0.720  PGD 0.673
  eps=2/255  FGSM 0.524  PGD 0.357
  eps=4/255  FGSM 0.268  PGD 0.064
  eps=8/255  FGSM 0.105  PGD 0.000

frozen_noisy: clean 0.883
  eps=1/255  FGSM 0.740  PGD 0.698
  eps=2/255  FGSM 0.608  PGD 0.486
  eps=4/255  FGSM 0.383  PGD 0.137
  eps=8/255  FGSM 0.142  PGD 0.000

soft_noisy: clean 0.903
  eps=1/255  FGSM 0.782  PGD 0.747
  eps=2/255  FGSM 0.658  PGD 0.563
  eps=4/255  FGSM 0.435  PGD 0.204
  eps=8/255  FGSM 0.218  PGD 0.013

pretrain15: clean 0.935
  eps=1/255  FGSM 0.674  PGD 0.569
  eps=2/255  FGSM 0.433  PGD 0.187
  eps=4/255  FGSM 0.205  PGD 0.004
  eps=8/255  FGSM 0.083  PGD 0.000

pretrain30: clean 0.932
  eps=1/255  FGSM 0.720  PGD 0.638
  eps=2/255  FGSM 0.486  PGD 0.248
  eps=4/255  FGSM 0.253  PGD 0.013
  eps=8/255  FGSM 0.090  PGD 0.000


In [ ]:
def square_acc(clf, loader, eps, n_queries=1000):
    atk = torchattacks.Square(clf, norm='Linf', eps=eps,
                              n_queries=n_queries, n_restarts=1)
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        x_adv = atk(x, y)
        with torch.no_grad():
            correct += (clf(x_adv).argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total

sub = Subset(test, range(400))
aa_loader = DataLoader(sub, batch_size=128, shuffle=False, num_workers=8)

for name, clf in [("freeze", froz), ("freeze+noise", noisy)]:
    for eps in [2/255, 4/255]:
        print(f"{name} Square eps={eps*255:.0f}/255: {square_acc(clf, aa_loader, eps):.3f}")

freeze Square eps=2/255: 0.807
freeze Square eps=4/255: 0.677
freeze+noise Square eps=2/255: 0.900
freeze+noise Square eps=4/255: 0.887


In [12]:
sub = Subset(test, range(400))
aa_loader = DataLoader(sub, batch_size=128, shuffle=False, num_workers=8)

def eot_pgd(clf, x, y, eps, alpha, steps=20, eot=10):
    x0 = x.clone().detach()
    x_adv = (x0 + torch.empty_like(x0).uniform_(-eps, eps)).clamp(0, 1)
    for _ in range(steps):
        x_adv.requires_grad_(True)
        grad = torch.zeros_like(x_adv)
        for _ in range(eot):
            loss = F.cross_entropy(clf(x_adv), y)
            grad += torch.autograd.grad(loss, x_adv)[0]
        x_adv = x_adv.detach() + alpha * (grad / eot).sign()
        x_adv = torch.min(torch.max(x_adv, x0 - eps), x0 + eps).clamp(0, 1)
    return x_adv.detach()

def eot_acc(clf, loader, eps, reps=5):
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        x_adv = eot_pgd(clf, x, y, eps, alpha=eps/4)
        with torch.no_grad():
            logits = sum(clf(x_adv) for _ in range(reps)) / reps
            correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total


In [13]:
runs = [("soft", soft), ("soft_v2", soft_v2), ("soft_v3", soft_v3), ("frozen", noisy)]

print(f"{'model':12s} {'eps2':>7} {'eps4':>7}")
for name, clf in runs:
    a2 = eot_acc(clf, aa_loader, 2/255)
    a4 = eot_acc(clf, aa_loader, 4/255)
    print(f"{name:12s} {a2:7.3f} {a4:7.3f}")

model           eps2    eps4
soft           0.570   0.152
soft_v2        0.468   0.068
soft_v3        0.482   0.105
frozen         0.440   0.087


In [11]:
# curriculum froz под сильным PGD (steps=20, alpha=eps*2.5/20) — как vanilla 0.0522
for eps in [2/255, 4/255]:
    p = adv_acc(clf=froz, loader=sub_loader,
                atk=torchattacks.PGD(froz, eps=eps, alpha=eps*2.5/20,
                                     steps=20, random_start=True))
    print(f"frozen eps={eps*255:.0f}/255 PGD-20: {p:.3f}")

frozen eps=2/255 PGD-20: 0.346
frozen eps=4/255 PGD-20: 0.060
